<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Filtering in Frequency Domain</b></h1>
</div>

## Theoretical Foundations

This notebook formalizes the mathematical and algorithmic basis of the corresponding laboratory implementation. The section order mirrors the experimental workflow so that assumptions, estimation steps, diagnostics, and validation criteria remain directly traceable to the executable notebook.

### Technical Context

The Fourier transform decomposes spatial image structure into frequency components whose magnitude and phase encode different aspects of the image.

### Core Frequency-Domain Model

For image $f(x,y)$, the 2-D DFT produces $F(u,v)$. Filtering multiplies the spectrum by a transfer function $H(u,v)$, then reconstructs with the inverse transform: $g=\mathcal{F}^{-1}\{HF\}$.

### Notation and Conventions

| Symbol | Meaning |
| --- | --- |
| $f(x,y)$ | spatial-domain image |
| $F(u,v)$ | 2-D Fourier transform |
| $H(u,v)$ | frequency-domain filter |
| $G(u,v)$ | filtered spectrum |
| $D(u,v)$ | radial frequency distance |
| $D_0$ | cutoff frequency |

### Analytical Scope

Interpret spectra, distinguish magnitude/phase, design common frequency filters, explain ringing, remove periodic noise with notches, correct slowly varying illumination, and validate reconstruction/filtering.


## 1. Spatial Frequency

A sinusoidal brightness pattern can be written as:

$$
g(x)=A\sin(2\pi f x+\phi)
$$

where:

- $A$ = amplitude;
- $f$ = spatial frequency;
- $\phi$ = phase.

Low spatial frequency means intensity changes slowly across space.  
High spatial frequency means intensity changes rapidly.

**Important:** high frequency does not mean high brightness.


## 2. Sinusoids, Complex Numbers, and the DFT

The DFT of a 1-D signal is:

$$
X[k]
=
\sum_{n=0}^{N-1}
x[n]e^{-j2\pi kn/N}
$$

Inverse:

$$
x[n]
=
\frac{1}{N}
\sum_{k=0}^{N-1}
X[k]e^{j2\pi kn/N}
$$

Euler's identity:

$$
e^{j\theta}
=
\cos(\theta)+j\sin(\theta)
$$

For $X=a+jb$:

$$
|X|=\sqrt{a^2+b^2}
$$

and

$$
\phi=\operatorname{atan2}(b,a)
$$

Magnitude = frequency strength.  
Phase = spatial alignment.


## 3. The 2-D Fourier Transform for Images

For image $f(x,y)$:

$$
F(u,v)
=
\sum_{x=0}^{M-1}
\sum_{y=0}^{N-1}
f(x,y)
e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$

The FFT computes the DFT efficiently.

`fftshift` moves the zero-frequency component to the center:

- center → low frequencies;
- farther from center → high frequencies.

Before inverse FFT, undo the shift with `ifftshift`.


## 4. Reading a 2-D Spectrum

A useful orientation rule:

> Spatial stripes produce spectral energy perpendicular to the stripe direction.

Let's prove it visually.


## 5. Inverse FFT and Reconstruction

The Fourier transform is reversible if we keep all coefficients.


## 6. Magnitude vs Phase

Every Fourier coefficient can be written as:

$$
F(u,v)=|F(u,v)|e^{j\phi(u,v)}
$$

Magnitude tells us how strong a frequency is.  
Phase strongly controls spatial organization.


## 7. Frequency-Domain Filtering

Let $F$ be the image spectrum and $H$ the filter:

$$
G(u,v)=H(u,v)F(u,v)
$$

Then:

$$
g(x,y)=\mathcal{F}^{-1}\{G(u,v)\}
$$

Workflow:

1. FFT;
2. center with `fftshift`;
3. construct $H$;
4. multiply $H\cdot F$;
5. undo shift;
6. IFFT;
7. keep the real component.


## 8. Frequency Distance Grid

For circular filters:

$$
D(u,v)=
\sqrt{(u-u_0)^2+(v-v_0)^2}
$$


## 9. Ideal, Gaussian, and Butterworth Low-Pass Filters

### Ideal LPF

$$
H(u,v)=
\begin{cases}
1,&D(u,v)\le D_0\\
0,&D(u,v)>D_0
\end{cases}
$$

### Gaussian LPF

$$
H(u,v)
=
\exp\left(
-\frac{D(u,v)^2}{2D_0^2}
\right)
$$

### Butterworth LPF

$$
H(u,v)
=
\frac{1}
{1+\left(\frac{D(u,v)}{D_0}\right)^{2n}}
$$

Butterworth order $n$ controls transition steepness.


## 10. Ringing and the Gibbs Phenomenon

A hard spectral cutoff corresponds to an oscillatory spatial response:

> abrupt spectral boundary → spatial oscillations → halos near edges


## 11. High-Pass Filtering

For a normalized LPF:

$$
H_{HP}=1-H_{LP}
$$

High frequencies contain edges and fine detail, but can also contain noise.


## 12. High-Boost Sharpening

A pure high-pass result mainly contains detail.

For sharpening:

$$
g(x,y)=f(x,y)+k f_{HP}(x,y)
$$


## 13. Convolution Theorem

$$
f*h
\quad\Longleftrightarrow\quad
F\cdot H
$$

Spatial convolution corresponds to multiplication in the frequency domain.

### Circular vs Linear Convolution

A DFT assumes periodic extension.

Therefore direct FFT multiplication naturally performs **circular convolution**.  
For ordinary linear convolution, appropriate zero-padding is generally required.


## 14. Band-Pass and Band-Reject Filters

Band-pass keeps:

$$
D_1\le D(u,v)\le D_2
$$

Band-reject removes that interval.


## 15. Periodic Noise

Periodic interference is one of the strongest reasons to use the frequency domain.

Repeated interference often becomes isolated off-center peaks in the spectrum.


## 16. Spectral Peak Detection

The following detector is intentionally simple:

1. remove the central low-frequency area;
2. rank remaining coefficients;
3. keep strong points separated by a minimum distance.


## 17. Notch-Reject Filtering

A notch-reject filter suppresses a small neighborhood around selected unwanted frequencies.

Real images have conjugate-symmetric spectra, so corresponding symmetric frequencies must also be considered.


## 18. Moiré Removal

Moiré is a repeated interference pattern. It can often be easier to isolate in the Fourier domain than in the spatial domain.


## 19. Slowly Varying Illumination / Shading

A simple multiplicative model is:

$$
I(x,y)\approx R(x,y)L(x,y)
$$

where:

- $R$ = reflectance / useful structure;
- $L$ = slowly varying illumination.

Because illumination varies slowly, it is dominated by low frequencies.


## 20. Cutoff Sensitivity

For a low-pass filter:

- smaller cutoff → stronger smoothing;
- larger cutoff → more detail preserved.


## 21. Quantitative Checks

MSE:

$$
\mathrm{MSE}
=
\frac{1}{MN}
\sum_{x,y}
[f(x,y)-g(x,y)]^2
$$

PSNR:

$$
\mathrm{PSNR}
=
10\log_{10}
\left(
\frac{255^2}{\mathrm{MSE}}
\right)
$$

PSNR measures numerical fidelity to a reference. It is not a universal perceptual-quality metric.


## 22. Validation Checks

This section develops the frequency-domain theory for validation checks.


## 23. Failure Modes and Diagnostic Signatures

The following implementation failures have distinct diagnostic signatures:

- raw FFT magnitude obscures weak spectral components because of extreme dynamic range;
- missing `fftshift` / `ifftshift` misaligns the designed transfer function with the spectrum;
- confusing luminance level with spatial frequency leads to incorrect filter interpretation;
- aggressive Ideal cutoffs introduce ringing in the spatial domain;
- high-pass filtering can amplify acquisition noise together with fine structure;
- arbitrary suppression of bright spectral peaks can remove valid periodic texture;
- ignoring conjugate symmetry can produce inconsistent notch designs for real-valued images;
- insufficient zero-padding causes circular-convolution artifacts;
- PSNR alone does not establish perceptual or task-level improvement.

These signatures are used as diagnostic evidence during filter design and result review.

## 24. Parameter Sensitivity and Controlled Experiments

The laboratory uses controlled parameter studies rather than open-ended exercises.

### Spectrum Characterization

- generate gratings at multiple spatial frequencies and verify the predicted FFT peak locations;
- compare horizontal, vertical, and diagonal structures;
- verify numerical consistency of FFT → IFFT reconstruction.

### Filter Sensitivity

- evaluate cutoff values $D_0\in\{10,20,40,80\}$;
- evaluate Butterworth orders $n\in\{1,2,4,8\}$;
- quantify and visualize the smoothing/ringing trade-off;
- evaluate the effect of high-pass filtering on both detail and noise.

### Periodic-Interference Suppression

- compare manually selected and automatically detected notch locations;
- study notch-radius sensitivity;
- evaluate moiré removal on `car-moire-pattern.tif`;
- evaluate low-frequency illumination correction on `text-spotshade.tif`;
- compare circular and zero-padded linear convolution behavior.

Each experiment changes one design variable while preserving the remaining processing configuration.

## 25. Method Selection and Technical Discussion

The frequency-domain method is selected according to the observed spectral structure and the spatial artifact produced after reconstruction.

Key engineering decisions are:

- `fftshift` is used only for centered interpretation and transfer-function design;
- logarithmic magnitude display is used for spectral diagnostics, not for filtering;
- Gaussian filtering is preferred when a smooth transition and minimal ringing are required;
- Butterworth filtering is used when transition steepness must be controlled explicitly;
- Ideal filtering is retained primarily as a reference because of its oscillatory spatial response;
- notch rejection is justified only when localized periodic peaks can be separated from valid image content;
- high-pass filtering is accepted only when the recovered detail outweighs amplified noise;
- zero-padding is required when frequency multiplication is intended to represent linear rather than circular convolution.

The principal filtering relation remains

$$
G(u,v)=H(u,v)F(u,v),
$$

with the validity of $H(u,v)$ determined by the resulting spatial-domain evidence.

## 26. Integrated Frequency-Domain Workflow

```text
Spatial image
    ↓
FFT2
    ↓
Centered spectrum
    ├── magnitude → energy / periodicity diagnostics
    └── phase     → spatial-structure preservation
    ↓
Transfer function H(u,v)
    ↓
Filtered spectrum G(u,v)
    ↓
Inverse shift + IFFT2
    ↓
Spatial reconstruction
    ↓
Quantitative and visual validation
```

This workflow is the common execution model for low-pass, high-pass, band, notch, moiré-removal, and illumination-correction experiments.

## Technical Synthesis

Frequency-domain processing is expressed by the transfer-function formulation

$$
G(u,v)=H(u,v)F(u,v),
$$

with reconstruction through the inverse Fourier transform. The complete analysis chain is

$$
\boxed{
\text{image}
\rightarrow
\mathcal{F}
\rightarrow
\text{spectrum interpretation}
\rightarrow
H(u,v)
\rightarrow
\mathcal{F}^{-1}
\rightarrow
\text{spatial result}
\rightarrow
\text{diagnostics}
}
$$

Filter family, cutoff, order, spectral localization, phase preservation, conjugate symmetry, ringing, and periodic-noise signatures determine whether a frequency-domain intervention is justified.

## Scope and Limitations

### Included

2-D DFT/IDFT, spectrum reading, magnitude/phase, low/high/band filters, ringing, convolution theorem, periodic-noise detection/removal, moiré, shading, sensitivity, and validation.

### Not included

Wavelets and advanced multiresolution methods.
